In [ ]:
library(terra)
library(ncdf4)
library(ggplot2)
library(tidyterra)

library(terra)
library(leaflet)

ncfile_path <- "C:/Users/qzhan/OneDrive - NIOZ/Attachments/01_LTER-LIFE/03_Model/3D_models_WaddenSea/Input/"

# --- Load topo (to get grid and Wadden island mask) ---
nc_topo <- nc_open(paste0(ncfile_path, "topo_adjusted_dws_200m_2009.nc"))
lonc <- ncvar_get(nc_topo, "lonc")
latc <- ncvar_get(nc_topo, "latc")
bathy <- ncvar_get(nc_topo, "bathymetry")
nc_close(nc_topo)

dim_xc <- dim(lonc)[1]
dim_yc <- dim(lonc)[2]

# Wadden island mask: TRUE = island (NA in topo)
island_mask <- is.na(bathy)

# --- Load Wadden Sea measurements ---
silt_nc <- rast(paste0(ncfile_path, "sediment_mud_fraction.nc"))

# Assign CRS (assume lon/lat, EPSG:4326)
if (crs(silt_nc) == "") crs(silt_nc) <- "EPSG:4326"

# Replace negative values with NA
silt_nc[silt_nc < 0] <- NA

# --- Load North Sea porosity ---
ns_nc <- rast(paste0(ncfile_path, "Ben_Sedprop.nc"))
crs(ns_nc)
ns_ll <- project(ns_nc, "EPSG:4326")  # reproject to lon/lat if needed

# --- Prepare target coordinates for GETM grid ---
xy <- cbind(as.vector(lonc), as.vector(latc))  # N x 2